# DynaLoRA revision experiments (Colab, GPU)

Runs the additional analyses of the revision on top of the results of `colab_run_all.ipynb` (same Drive folder): a training-strength sweep, teacher-forced likelihood of the sweep adapters and of the negative-only adapter on negative reviews, the plain adapter on a finer temperature grid for continuous entropy matching, re-scoring of all generations with a fluency model from another family and a unified domain classifier, and zero-shot benchmarks across lambda for both forms of scaling. Every step skips what is already on Drive, so the notebook can be re-run after a disconnection. Expected time on a T4: about 2 to 3 hours.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!pip install -q -r requirements.txt lm-eval
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/dynalora_results'
import os; print(os.listdir(OUT))

## 1. Training-strength sweep (balanced IMDB, learning rate only)

In [ ]:
!python scripts/run_all.py --out $OUT/sweep --models gpt2 --configs imdb_bal__lr5e-06 imdb_bal__lr1.5e-05 imdb_bal__lr5e-05 imdb_bal__lr1e-04 --methods dynalora --clf_configs imdb_bal yelp pubmed --n_samples 20

## 2. Teacher-forced likelihood: sweep adapters, and IMDB adapters on negative reviews

In [ ]:
!python scripts/likelihood_tradeoff.py --model gpt2 --adapters_dir $OUT/sweep/adapters --configs imdb_bal__lr5e-06 imdb_bal__lr1.5e-05 imdb_bal__lr5e-05 imdb_bal__lr1e-04 --out $OUT/likelihood
!python scripts/likelihood_tradeoff.py --model gpt2 --adapters_dir $OUT/gpt2/adapters --configs imdb_neg imdb_bal --sets domain_neg --out $OUT/likelihood
!python scripts/likelihood_summary.py $OUT/likelihood > /dev/null

## 3. Plain adapter on a finer temperature grid

In [ ]:
!python scripts/run_all.py --out $OUT/temps_gpt2 --adapters_dir $OUT/gpt2/adapters --models gpt2 --configs imdb_neg imdb_bal imdb_bal_nohtml yelp pubmed --methods lora_temp --temps 0.65 0.55 0.45 0.35 0.25 --n_samples 20
!python scripts/run_all.py --out $OUT/temps_qwen --adapters_dir $OUT/qwen/adapters --models Qwen/Qwen2.5-0.5B --configs imdb_bal --methods lora_temp --temps 0.65 0.55 0.45 0.35 0.25 --n_samples 20

## 4. Re-scoring with a fluency model from another family and a unified domain classifier

In [ ]:
!python scripts/rescore.py --inputs "$OUT/*/generations_all_metrics.csv" --out $OUT/rescore

## 5. Zero-shot benchmarks across lambda

In [ ]:
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir $OUT/gpt2/adapters --configs imdb_neg imdb_bal imdb_bal_nohtml yelp pubmed --out $OUT/bench
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-0.5B --adapters_dir $OUT/qwen/adapters --configs imdb_bal --out $OUT/bench --batch_size 8

## 6. Package the new results

In [ ]:
!cd $OUT && zip -qr /content/dynalora_revision.zip sweep temps_gpt2 temps_qwen rescore bench likelihood -x '*/adapters/*'
from google.colab import files
files.download('/content/dynalora_revision.zip')